# Goal: statistical-summary 

In [1]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\n'

In [2]:
import polars as pl

In [3]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-14\data\lev-14_merged.parquet"
pdf = pl.scan_parquet(path)

In [4]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('SECTION', String),
        ('ITEM_CODE', String),
        ('VALUE_RS', Int64),
        ('MULTIPLIER', Int64)])

# Useful Variables

In [5]:

cols = [
'SECTION',
'ITEM_CODE',
'VALUE_RS',
'MULTIPLIER',
]

In [6]:
df = pdf.select(cols)

In [7]:
df.head(2).collect()

SECTION,ITEM_CODE,VALUE_RS,MULTIPLIER
str,str,i64,i64
"""5.1""","""139""",55,21114
"""5.1""","""139""",65,33338


In [8]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in cols]
)

In [9]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

SECTION,ITEM_CODE,VALUE_RS,MULTIPLIER
u32,u32,u32,u32
1,42,31887,23567


# Logic

In [10]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_22064\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [11]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
ITEM_CODE,16593138.0,0.0,387.860975,175.743817,99.0,239.0,379.0,519.0,899.0
VALUE_RS,16593138.0,0.0,2076.884329,10113.083024,0.0,160.0,442.0,1300.0,4511750.0
MULTIPLIER,16593138.0,0.0,109948.758565,77923.766241,369.0,54218.0,113220.0,149973.0,2366902.0


# Categorical Columns

In [12]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))

SECTION


SECTION,count
i32,u32
null,16593138
